# TejaLens — Module 4 Training

**Before running:** Runtime → Change runtime type → T4 GPU → Save

**Run cells in order. Every cell is idempotent — safe to re-run after a restart.**

| Cell | What it does | Must re-run after restart? |
|------|-------------|---------------------------|
| 1 | Mount Drive + verify GPU | ✅ Yes |
| 2 | Clone/pull repo + install deps | ✅ Yes |
| 3 | Download ISIC 2018 seg data | Only if not cached on Drive |
| 4 | Train U-Net VGG16 (segmentation) | Only if weight missing from Drive |
| 5 | Download HAM10000 | Only if not cached on Drive |
| 6 | Train Swin-Small (research classifier) | Only if weight missing from Drive |
| 7 | Train EfficientNet-B0 (on-device classifier) | Only if weight missing from Drive |
| 8 | Download ISIC 2019 | Only if not cached on Drive |
| 9 | Fine-tune EfficientNet-B0 on ISIC 2019 | Only if weight missing from Drive |
| 10 | Save results JSON to Drive | ✅ After all training done |

In [ ]:
# Cell 1 — Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch, os
assert torch.cuda.is_available(), 'NO GPU — Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

# Create Drive output folder once
os.makedirs('/content/drive/MyDrive/tejalens', exist_ok=True)
print('Drive folder ready:', os.listdir('/content/drive/MyDrive/tejalens'))

In [ ]:
# Cell 2 — Clone/pull repo + install deps
import subprocess, os, sys

REPO = 'https://github.com/Durva-3124/Skin_Lesion_Detection.git'
REPO_DIR = '/content/tejalens'

if os.path.exists(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)
    print('Pulled latest changes')
else:
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
    print('Cloned repo')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install timm -q')
print('src importable:', os.path.isdir(f'{REPO_DIR}/src'))

In [ ]:
# Cell 3 — Download ISIC 2018 segmentation data
# Skips automatically if already downloaded
import zipfile, pathlib, requests
from tqdm import tqdm

IMG_DIR  = pathlib.Path('/content/data/isic2018_seg/ISIC2018_Task1-2_Training_Input')
MASK_DIR = pathlib.Path('/content/data/isic2018_seg/ISIC2018_Task1_Training_GroundTruth')

if len(list(IMG_DIR.glob('*.jpg'))) == 2594:
    print(f'Already present — Images: 2594, Masks: {len(list(MASK_DIR.glob("*.png")))}')
else:
    BASE = 'https://isic-challenge-data.s3.amazonaws.com'
    DEST = pathlib.Path('/content/data/isic2018_seg')
    for url in [
        f'{BASE}/2018/ISIC2018_Task1-2_Training_Input.zip',
        f'{BASE}/2018/ISIC2018_Task1_Training_GroundTruth.zip',
    ]:
        DEST.mkdir(parents=True, exist_ok=True)
        fname = url.split('/')[-1]
        out_path = DEST / fname
        print(f'Downloading {fname}...')
        r = requests.get(url, stream=True, timeout=600)
        with open(out_path, 'wb') as f, tqdm(total=int(r.headers.get('content-length', 0)), unit='B', unit_scale=True) as bar:
            for chunk in r.iter_content(65536):
                f.write(chunk); bar.update(len(chunk))
        with zipfile.ZipFile(out_path) as z:
            z.extractall(DEST)
        out_path.unlink()

print(f'Images: {len(list(IMG_DIR.glob("*.jpg")))}, Masks: {len(list(MASK_DIR.glob("*.png")))}')

In [ ]:
# Cell 4 — Train U-Net VGG16 segmentation (15 epochs, peaks at ~epoch 15)
# Skips if weight already saved to Drive
import pathlib, os

SAVE_PATH = '/content/drive/MyDrive/tejalens/unet_vgg16.pth'

if os.path.exists(SAVE_PATH):
    print(f'Already trained — {SAVE_PATH} exists. Delete from Drive to retrain.')
else:
    # Clear any stale cached imports
    import sys
    for k in list(sys.modules.keys()):
        if 'src' in k: del sys.modules[k]

    import src.segmentation.train as seg_train

    seg_train.SEG_DIR  = pathlib.Path('/content/data/isic2018_seg')
    seg_train.IMG_DIR  = seg_train.SEG_DIR / 'ISIC2018_Task1-2_Training_Input'
    seg_train.MASK_DIR = seg_train.SEG_DIR / 'ISIC2018_Task1_Training_GroundTruth'

    seg_train.train(
        encoder='vgg16', epochs=15, lr=1e-4,
        image_size=256, batch_size=8,
        save_path=SAVE_PATH
    )
    print(f'\n✅ Saved to Drive: {SAVE_PATH}')

In [ ]:
# Cell 5 — Download HAM10000 from Harvard Dataverse (no auth required)
# Skips automatically if already downloaded
import zipfile, pathlib, requests
from tqdm import tqdm

DEST = pathlib.Path('/content/data/ham10000')
DEST.mkdir(parents=True, exist_ok=True)

if len(list(DEST.glob('*.jpg'))) >= 10000:
    print(f'Already present — {len(list(DEST.glob("*.jpg")))} images')
else:
    FILES = [
        {'id': 4338392, 'name': 'HAM10000_metadata.tab', 'zip': False},
        {'id': 3172585, 'name': 'HAM10000_images_part_1.zip', 'zip': True},
        {'id': 3172584, 'name': 'HAM10000_images_part_2.zip', 'zip': True},
    ]
    BASE = 'https://dataverse.harvard.edu/api/access/datafile'
    for f in FILES:
        dest_path = DEST / f['name']
        if dest_path.exists():
            print(f'Already exists: {f["name"]}')
            continue
        print(f'Downloading {f["name"]}...')
        r = requests.get(f'{BASE}/{f["id"]}', headers={'User-Agent': 'Mozilla/5.0'}, stream=True, timeout=300)
        with open(dest_path, 'wb') as out, tqdm(total=int(r.headers.get('content-length', 0)), unit='B', unit_scale=True) as bar:
            for chunk in r.iter_content(65536):
                out.write(chunk); bar.update(len(chunk))
        if f['zip']:
            with zipfile.ZipFile(dest_path) as z:
                z.extractall(DEST)
            dest_path.unlink()

print(f'HAM10000 images: {len(list(DEST.glob("*.jpg")))}')

In [ ]:
# Cell 6 — Train Swin-Small on HAM10000 (research/benchmark classifier)
# Skips if weight already saved to Drive
import os, sys

SAVE_PATH = '/content/drive/MyDrive/tejalens/swin_small_ham10000.pth'

if os.path.exists(SAVE_PATH):
    print(f'Already trained — {SAVE_PATH} exists. Delete from Drive to retrain.')
else:
    for k in list(sys.modules.keys()):
        if 'src' in k: del sys.modules[k]

    from src.classification.train import train as train_cls

    _, swin_metrics = train_cls(
        model_name='swin_small',
        dataset_name='ham10000',
        epochs=30, lr=1e-4, batch_size=32,
        loss_type='focal',
        save_path=SAVE_PATH
    )
    print('\n✅ Swin-Small metrics:', swin_metrics)

In [ ]:
# Cell 7 — Train EfficientNet-B0 on HAM10000 (on-device classifier)
# Skips if weight already saved to Drive
import os, sys

SAVE_PATH = '/content/drive/MyDrive/tejalens/efficientnet_b0_ham10000.pth'

if os.path.exists(SAVE_PATH):
    print(f'Already trained — {SAVE_PATH} exists. Delete from Drive to retrain.')
else:
    for k in list(sys.modules.keys()):
        if 'src' in k: del sys.modules[k]

    from src.classification.train import train as train_cls

    _, b0_metrics = train_cls(
        model_name='efficientnet_b0',
        dataset_name='ham10000',
        epochs=30, lr=1e-4, batch_size=32,
        loss_type='focal',
        save_path=SAVE_PATH
    )
    print('\n✅ EfficientNet-B0 HAM10000 metrics:', b0_metrics)

In [ ]:
# Cell 8 — Download ISIC 2019
# Skips automatically if already downloaded
import zipfile, pathlib, requests
from tqdm import tqdm

DEST = pathlib.Path('/content/data/isic2019')
IMG_DIR = DEST / 'ISIC_2019_Training_Input'

if len(list(IMG_DIR.glob('*.jpg'))) >= 25000:
    print(f'Already present — {len(list(IMG_DIR.glob("*.jpg")))} images')
else:
    BASE = 'https://isic-challenge-data.s3.amazonaws.com'
    DEST.mkdir(parents=True, exist_ok=True)
    for url in [
        f'{BASE}/2019/ISIC_2019_Training_Input.zip',
        f'{BASE}/2019/ISIC_2019_Training_GroundTruth.csv',
    ]:
        fname = url.split('/')[-1]
        out_path = DEST / fname
        print(f'Downloading {fname}...')
        r = requests.get(url, stream=True, timeout=600)
        with open(out_path, 'wb') as f, tqdm(total=int(r.headers.get('content-length', 0)), unit='B', unit_scale=True) as bar:
            for chunk in r.iter_content(65536):
                f.write(chunk); bar.update(len(chunk))
        if fname.endswith('.zip'):
            with zipfile.ZipFile(out_path) as z:
                z.extractall(DEST)
            out_path.unlink()

print(f'ISIC2019 images: {len(list(IMG_DIR.glob("*.jpg")))}')

In [ ]:
# Cell 9 — Fine-tune EfficientNet-B0 on ISIC 2019 (8 classes)
# Skips if weight already saved to Drive
import os, sys

SAVE_PATH = '/content/drive/MyDrive/tejalens/efficientnet_b0_isic2019.pth'

if os.path.exists(SAVE_PATH):
    print(f'Already trained — {SAVE_PATH} exists. Delete from Drive to retrain.')
else:
    for k in list(sys.modules.keys()):
        if 'src' in k: del sys.modules[k]

    from src.classification.train import train as train_cls

    _, b0_19_metrics = train_cls(
        model_name='efficientnet_b0',
        dataset_name='isic2019',
        epochs=20, lr=5e-5, batch_size=32,
        loss_type='focal',
        save_path=SAVE_PATH
    )
    print('\n✅ EfficientNet-B0 ISIC2019 metrics:', b0_19_metrics)

In [ ]:
# Cell 10 — Collect all results and save to Drive
import json, pathlib, os, torch

DRIVE = pathlib.Path('/content/drive/MyDrive/tejalens')

# Load metrics from Drive-saved checkpoints and report what exists
weights = [
    'unet_vgg16.pth',
    'swin_small_ham10000.pth',
    'efficientnet_b0_ham10000.pth',
    'efficientnet_b0_isic2019.pth',
]
print('Weights saved to Drive:')
for w in weights:
    path = DRIVE / w
    size = os.path.getsize(path) / 1e6 if path.exists() else None
    status = f'{size:.1f} MB' if size else 'MISSING'
    print(f'  {w}: {status}')

# Save metrics collected during this session (if variables exist)
results = {}
for name, var in [('swin_small_ham10000', 'swin_metrics'),
                   ('efficientnet_b0_ham10000', 'b0_metrics'),
                   ('efficientnet_b0_isic2019', 'b0_19_metrics')]:
    if var in dir():
        results[name] = eval(var)

if results:
    out = DRIVE / 'module4_results.json'
    with open(out, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'\nResults saved to {out}')
    print(json.dumps(results, indent=2, default=str))